# Day 1 Overnight Corpus Generation (Colab Pro+ A100)

Run **after** `day1_smoke.ipynb` is green. Generates 250 trajectories, distributed 50/100/100 across L0/L1/L2 per `configs/default.yaml`.

Workflow:
1. Mount Drive, clone/pull repo, install deps.
2. Kick off `scripts/01_generate_corpus.py` as a background process logging to Drive.
3. Watch the log for the first ~10 trajectories to confirm rate.
4. Close laptop. Resume from JSONL on disconnect via `--resume`.

Decision rule: if rate after the first 10 trajectories implies >10h to finish, kill the run and re-launch with `--n_trajectories 200` (still well above the 150 threshold from §12).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/agent_faithfulness'
REPO_DIR  = '/content/agent_faithfulness'
GITHUB_URL = 'https://github.com/<YOUR_USERNAME>/agent_faithfulness.git'  # <-- EDIT ME

import os, subprocess
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/activations', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', GITHUB_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])

%cd $REPO_DIR
!pip install -q -r requirements.txt

In [ ]:
# Launch the corpus generation as a background process.
import time, os

TIMESTAMP = int(time.time())
LOG_PATH = f'{DRIVE_ROOT}/logs/gen_{TIMESTAMP}.log'
OUT_DIR  = f'{DRIVE_ROOT}/data'

cmd = (
    f'cd {REPO_DIR} && nohup python scripts/01_generate_corpus.py '
    f'--config configs/default.yaml '
    f'--n_trajectories 250 '
    f'--out_dir {OUT_DIR} '
    f'--resume '
    f'> {LOG_PATH} 2>&1 &'
)
print('Launching:', cmd)
os.system(cmd)
print('Log path:', LOG_PATH)

In [ ]:
# Watch the first ~10 trajectories before walking away.
import time, os, json
from pathlib import Path

JSONL = Path(f'{DRIVE_ROOT}/data/trajectories.jsonl')

for _ in range(40):  # ~40 minutes of observation max
    n = 0
    if JSONL.exists():
        with open(JSONL) as f:
            n = sum(1 for _ in f)
    print(f't+{int(time.time())%100000:5d}  trajectories on disk: {n}')
    if n >= 10:
        break
    time.sleep(60)

# Tail log
print('\n--- last 30 log lines ---')
os.system(f'tail -n 30 {LOG_PATH}')

In [ ]:
# (Optional) Kill the run if rate is too slow; re-launch with fewer trajectories.
# !pkill -f 01_generate_corpus.py